In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Read Data from Bronze Layer**

In [0]:
df_bronze = spark.read.table("fmcg.bronze.orders")

In [0]:
display(df_bronze.limit(5))

### **Transformations**

**1. Keep the records which is having order_qty**

In [0]:
print(df_bronze.count(), df_bronze.select("order_qty").count())

In [0]:
df_silver = df_bronze.filter(F.col("order_qty").isNotNull())

In [0]:
print(df_silver.count())

**2. Clean customer_id -> Keep  numeric, else set to 99999999**

In [0]:
display(df_silver.filter(~F.col("customer_id").rlike(r"^[0-9]+$")).select("customer_id").distinct())

In [0]:
df_silver = (
  df_silver.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike(r"^[0-9]+$"), F.col("customer_id"))
     .otherwise("99999999")
     .cast("string")
    )
)

In [0]:
display(df_silver.filter(~F.col("customer_id").rlike(r"^[0-9]+$")).select("customer_id").distinct())

**3. Clean order_placement_date - Remove weekday name from date text & make a uniform format**

In [0]:
display(df_silver.select("order_placement_date").distinct().limit(10))

In [0]:
df_silver = df_silver.withColumn(
  "order_placement_date",
  F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
  )

In [0]:
display(df_silver.select("order_placement_date").distinct().limit(10))

In [0]:
df_silver = df_silver.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy")
    )
)

In [0]:
display(df_silver.select("order_placement_date").distinct().limit(10))

**4. Drop duplicates**

In [0]:
df_silver = df_silver.dropDuplicates(["order_id", "order_placement_date", "order_qty", "product_id", "customer_id"])

**5. Convert product id to string**

In [0]:
df_silver = df_silver.withColumn("product_id", F.col("product_id").cast("string"))

**6. Check min and max dates**

In [0]:
display(df_silver.agg(F.min("order_placement_date").alias("min_date"), F.max("order_placement_date").alias("max_date")))

**7. Get the product_code**

In [0]:
df_products = spark.read.table("fmcg.silver.products")

display(df_products.limit(5))

In [0]:
df_joined = df_silver.join(df_products, on = "product_id", how = "inner").select(df_silver["*"], df_products["product_code"])
display(df_joined.limit(10))

In [0]:
df_silver = df_joined.select("product_code", "order_id", "customer_id", "product_id", "order_qty", "order_placement_date", "file_name", "file_size", "dataload_ts")

In [0]:
display(df_silver.limit(5))

In [0]:
if not (spark.catalog.tableExists("fmcg.silver.orders")):
    df_silver.write \
        .format("delta") \
            .option("delta.enableChangeDataFeed", True) \
                .option("mergeSchema", True) \
                    .mode("overwrite") \
                        .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

else:
    silver_delta_table = DeltaTable.forName(spark, f"{catalog}.{silver_schema}.{data_source}")
    
    silver_delta_table.alias("t").merge(df_silver.alias("s"), 
                                        "t.product_code == s.product_code AND  \
                                            t.order_id == s.order_id And \
                                                t.customer_id == s.customer_id AND\
                                                    t.order_placement_date == s.order_placement_date \
                                                    ") \
                                .whenMatchedUpdatedAll() \
                                    .whenNotMatchedInsertAll() \
                                        .execute()
